In [ ]:
import sys
print(sys.executable)

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./vectorstores/alice")
collection = client.get_collection("alice")

# 查看資料庫內所有資料
results = collection.get(include=["documents", "metadatas"])

print(f"共有 {len(results['ids'])} 筆資料")
for i, (id_, doc, meta) in enumerate(zip(results['ids'], results['documents'], results['metadatas'])):
    print(f"\n[{i+1}] id: {id_}")
    print(f"     type: {meta.get('type')}")
    print(f"     source: {meta.get('source')}")
    print(f"     document: {doc[:100]}")

In [ ]:
import os, sys
import torch
import chromadb
from sentence_transformers import SentenceTransformer

# 讓 Python 找得到 ModelTraining 資料夾內的 notemind 套件
MODEL_TRAINING_PATH = r"C:\Users\user\desktop\rag\ModelTraining"
if MODEL_TRAINING_PATH not in sys.path:
    sys.path.insert(0, MODEL_TRAINING_PATH)
from notemind.infer import NoteMind

# 有 GPU 就用 GPU，否則退回 CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用裝置: {device}")

# 檢索用多語嵌入模型（繁中+英文皆強，取代對中文較弱的 CLIP）
# E5 需要前綴：文件用 "passage: "，查詢用 "query: "
embedder = SentenceTransformer("intfloat/multilingual-e5-base", device=device)

def embed_query(text):
    return embedder.encode([f"query: {text}"], normalize_embeddings=True)[0].tolist()

# 連接對應使用者的資料庫
user_name = "alice"
client = chromadb.PersistentClient(path=f"./vectorstores/{user_name}")
vector_collection = client.get_collection(user_name)  # collection 名稱 = user_name

# 載入 NoteMind-LM（取代 gemini-2.5-flash）：完全離線、不需 API 金鑰，
# 權重與 tokenizer 來自 ModelTraining/checkpoints/
if "lm" not in globals():
    lm = NoteMind.load(device=device)


In [ ]:
CTX_TOKEN_BUDGET = 360  # 在 NoteMind 的 512 context window 內，預留空間給指令+問題+答案

def budget_notes(docs, budget=CTX_TOKEN_BUDGET):
    """依相關度保留檢索到的筆記片段，直到達到 token 預算，避免 prompt 溢出視窗
    而把開頭的作答指令截斷掉。"""
    kept, used = [], 0
    for d in docs:
        n = len(lm.tok.encode(d).ids)
        if kept and used + n > budget:
            break
        kept.append(d); used += n
    return kept

def rag_query(user_question, top_k=8):
    # Step 1: 用 multilingual-e5 embed 問題（query 前綴）
    q_embedding = embed_query(user_question)

    # Step 2: 搜尋相關文件片段（筆記在嵌入時已切塊，每塊較短）
    results = vector_collection.query(
        query_embeddings=[q_embedding],
        n_results=top_k
    )

    # Step 3: 取出片段並依 token 預算裁切，確保塞得進 NoteMind 的視窗
    docs = budget_notes(results["documents"][0])

    # Step 4: 交給 NoteMind-LM 根據檢索到的筆記作答
    answer = lm.answer(docs, user_question)
    return answer

input_question = input('輸入問題: ')
answer = rag_query(
    user_question=input_question,
    top_k=8
)
print(answer)
